In [0]:
import concurrent.futures
import datetime as dt
import hashlib
import html.parser
import pathlib
import re
import shutil
import unicodedata
import urllib.parse

import requests
from pyspark.sql import functions as F

ANIOS = range(2017, dt.date.today().year + 1)
PAGINA = "https://www.salud.gob.ec/gacetas-vectoriales-{anio}/"
LANDING = pathlib.Path("/Volumes/vigia/bronze/landing/gacetas")
TABLA_INDICE = "vigia.bronze.gacetas_indice"
AGENTE = {"User-Agent": "vigia-dengue/0.1 (proyecto de portafolio sin fines comerciales)"}

In [0]:
def simple(texto):
    """Mayúsculas y sin tildes"""
    return unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode().upper()


def es_otro_boletin(texto, url):
    """En el 2025 existen gacetas que hacen referencia a la fiebre amarilla, al no ser relevantes, se las marca para excluirlas"""
    t = simple(f"{texto} {url}")
    return "AMARILLA" in t and "VECTORIAL" not in t


class EnlacesPDF(html.parser.HTMLParser):
    """Anota cada enlace a un PDF con su texto y con el texto que hay entre él y el enlace anterior"""

    def __init__(self):
        super().__init__()
        self.enlaces, self._antes, self._actual = [], [], None

    def handle_starttag(self, tag, attrs):
        href = (dict(attrs).get("href") or "").strip()
        if tag == "a" and href.lower().split("?")[0].endswith(".pdf"):
            self._actual = {"url": href, "texto": [], "antes": " ".join(self._antes)}

    def handle_data(self, data):
        (self._actual["texto"] if self._actual else self._antes).append(data)

    def handle_endtag(self, tag):
        if tag == "a" and self._actual:
            self._actual["texto"] = " ".join(self._actual["texto"])
            self.enlaces.append(self._actual)
            self._actual, self._antes = None, []


SE = re.compile(r"\bSE\s*[-–—_]?\s*(\d{1,2})\b")


def semana(enlace):
    """La SE viene en el texto del enlace o, si dice 'Click aquí', en el título que lo precede"""
    en_texto, antes = SE.findall(enlace["texto"]), SE.findall(enlace["antes"])
    if en_texto:
        return int(en_texto[0])
    return int(antes[-1]) if antes else None


def indice_del_anio(anio):
    pagina = PAGINA.format(anio=anio)
    respuesta = requests.get(pagina, headers=AGENTE, timeout=60)
    respuesta.raise_for_status()
    lector = EnlacesPDF()
    lector.feed(respuesta.content.decode("utf-8", errors="replace"))  # las páginas del MSP están en UTF-8
    filas, vistos = [], set()
    for enlace in lector.enlaces:
        url = urllib.parse.urljoin(pagina, enlace["url"])
        url = re.sub(r"^http://(www\.)?salud\.gob\.ec", "https://www.salud.gob.ec", url)  # enlaces viejos
        if url not in vistos:  # algunos enlaces se repiten partidos en dos
            vistos.add(url)
            filas.append({"anio": anio, "se": semana(enlace), "url": url,
                          "excluir": es_otro_boletin(enlace["texto"], url)})
    return filas


indice = []
for anio in ANIOS:
    try:
        indice += indice_del_anio(anio)
    except requests.RequestException as error:
        print(f"{anio}: no se pudo leer la página ({error})")
excluidos = sum(fila["excluir"] for fila in indice)
print(f"{len(indice)} enlaces en el índice: {len(indice) - excluidos} gacetas y {excluidos} boletines excluidos")

In [0]:
def nombre_archivo(fila):
    """SE42__ETV_Gaceta_42.pdf: la semana al comienzo y el nombre original después"""
    original = re.sub(r"[^\w.-]", "_", pathlib.PurePosixPath(urllib.parse.urlparse(fila["url"]).path).name)
    return f"SE{fila['se'] or 0:02d}__{original}"


def descargar(fila):
    if fila["excluir"]:  # boletín de otra enfermedad: queda en el índice con su URL, pero no se baja
        return {**fila, "archivo": None, "estado": "excluido: boletín de fiebre amarilla",
                "bytes": None, "sha256": None}
    carpeta = LANDING / f"anio={fila['anio']}"
    destino = carpeta / nombre_archivo(fila)
    if destino.exists():
        contenido, estado = destino.read_bytes(), "ya estaba"
    else:
        try:
            respuesta = requests.get(fila["url"], headers=AGENTE, timeout=120)
            respuesta.raise_for_status()
            contenido = respuesta.content
        except requests.RequestException as error:
            return {**fila, "archivo": None, "estado": f"ERROR {type(error).__name__}: {error}"[:200],
                    "bytes": None, "sha256": None}
        if b"%PDF" not in contenido[:1024]:
            return {**fila, "archivo": None, "estado": "ERROR: la respuesta no es un PDF",
                    "bytes": len(contenido), "sha256": None}
        carpeta.mkdir(parents=True, exist_ok=True)
        parcial = carpeta / f"{destino.name}.parcial"
        parcial.write_bytes(contenido)
        shutil.move(parcial, destino)
        estado = "descargado"
    return {**fila, "archivo": destino.name, "estado": estado, "bytes": len(contenido),
            "sha256": hashlib.sha256(contenido).hexdigest()}


# Cuatro descargas a la vez
with concurrent.futures.ThreadPoolExecutor(max_workers=4) as hilos:
    resultados = list(hilos.map(descargar, indice))

# El índice queda como tabla: sirve para el reporte de semanas faltantes y para rastrear cada PDF a su URL.
# La clave excluir no pasa a la tabla: el estado excluido: ... ya lo dice.
esquema = "anio int, se int, url string, archivo string, estado string, bytes long, sha256 string"
tabla = spark.createDataFrame(resultados, esquema).withColumn("listado_en", F.current_date())
tabla.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLA_INDICE)

In [0]:
def inicio_semana_1(anio):
    """Domingo en que empieza la SE 1: la semana que contiene el 4 de enero"""
    cuatro = dt.date(anio, 1, 4)
    return cuatro - dt.timedelta(days=(cuatro.weekday() + 1) % 7)


def semanas_del_anio(anio):
    return (inicio_semana_1(anio + 1) - inicio_semana_1(anio)).days // 7


resumen = []
for anio in ANIOS:
    del_anio = [r for r in resultados if r["anio"] == anio]
    presentes = {r["se"] for r in del_anio if r["archivo"]}
    hasta = semanas_del_anio(anio) if anio < dt.date.today().year else max(presentes, default=0)
    faltan = sorted(set(range(1, hasta + 1)) - presentes)
    resumen.append((
        anio,
        sum(not r["excluir"] for r in del_anio),
        sum(r["excluir"] for r in del_anio),
        sum(r["estado"] == "descargado" for r in del_anio),
        sum(r["estado"].startswith("ERROR") for r in del_anio),
        # Sin semana y sin excluir: lo único que hay que revisar a mano.
        sum(r["se"] is None and not r["excluir"] for r in del_anio),
        ", ".join(map(str, faltan)),
    ))

display(spark.createDataFrame(
    resumen,
    "anio int, gacetas int, excluidas int, descargadas_ahora int, errores int, sin_se int, semanas_faltantes string"))